# Scrape antifakenewscenter.com – #มิจฉาชีพ news

Keep only posts tagged **#มิจฉาชีพ** together with **#อาชญากรรมออนไลน์** and/or **#ข่าวปลอม**, then
label them. Run the cells from top to bottom; the last cell saves the dataset.

The site is WordPress, so instead of parsing the tag page HTML we read the same posts of the
`/tag/มิจฉาชีพ/` archive through its public REST API (`/wp-json/wp/v2/posts?tags=<id>`). Each post
already carries its tag IDs, so no per-post page fetch is needed.

Output columns, in order:

| Columns | Made by |
|---|---|
| `post_id`, `date`, `hashtag`, `title`, `content` | the scrape (`date` is `DD/MM/YYYY` in the CSV) |
| `Channel` | keyword rule: `call` if the scammer phones the victim, otherwise `text` |
| `ORG` | keyword match against a list of Thai organizations (see "Organization column") |
| `scam_type`, `method_channel` | Qwen, first pass (see "Scam type and channel columns") |
| `call_suitable`, `summary`, `caller_role`, `goal`, `tactics`, `phrases`, `scenario_key`, `scenario_group` | Qwen, second pass (see "Scenario columns") |
| `source_url` | built from `post_id` |

`Channel` and `ORG` come from keyword rules, so they are a first pass, not ground truth.

The last cell saves everything to `data/antifakenews_scam.csv`.

In [ ]:
import html
import io
import re
import time
from collections import Counter
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

API = "https://www.antifakenewscenter.com/wp-json/wp/v2"
HEADERS = {"User-Agent": "Mozilla/5.0 (research scraper; Callas project)"}
DELAY = 1.0  # seconds between requests

BASE_TAG = "มิจฉาชีพ"
TARGET_TAGS = ["อาชญากรรมออนไลน์", "ข่าวปลอม"]

# Test run: a number (e.g. 10) fetches only the first page of the API and keeps that many of the
# newest posts, and saves to a separate file so the real dataset is not overwritten.
# None = the full run: every page, every post, saved to data/antifakenews_scam.csv.
MAX_POSTS = 10
OUT_PATH = Path("data/antifakenews_scam.csv" if MAX_POSTS is None else "data/antifakenews_scam_test.csv")

In [ ]:
def get_json(path, params, retries=3):
    """GET an API endpoint with retries; returns (json, response headers)."""
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(f"{API}/{path}", params=params, headers=HEADERS, timeout=30)
            r.raise_for_status()
            return r.json(), r.headers
        except requests.RequestException:
            if attempt == retries:
                raise
            time.sleep(2 * attempt)


def find_tag_id(name):
    """Resolve a tag name (with or without a leading '#') to its WordPress tag ID."""
    tags, _ = get_json("tags", {"search": name, "per_page": 100, "_fields": "id,name,count"})
    exact = [t for t in tags if t["name"].lstrip("#").strip() == name]
    if not exact:
        raise ValueError(f"tag not found: {name}")
    return max(exact, key=lambda t: t["count"])["id"]


tag_ids = {name: find_tag_id(name) for name in [BASE_TAG, *TARGET_TAGS]}
tag_ids

In [ ]:
def fetch_posts(tag_id, max_pages=None):
    """All posts of a tag (same set as the /tag/<slug>/ archive pages), newest first.
    max_pages stops early: 1 page is 100 posts, enough for a test run."""
    posts, page, total_pages = [], 1, 1
    while page <= total_pages and (max_pages is None or page <= max_pages):
        batch, headers = get_json("posts", {"tags": tag_id, "per_page": 100, "page": page,
                                            "_fields": "id,date,title,content,tags"})
        total_pages = int(headers["X-WP-TotalPages"])
        posts += batch
        print(f"page {page}/{total_pages}: {len(posts)}/{headers['X-WP-Total']} posts")
        page += 1
        time.sleep(DELAY)
    return posts


raw_posts = fetch_posts(tag_ids[BASE_TAG], max_pages=None if MAX_POSTS is None else 1)

In [ ]:
FOOTER_HEADING = "ช่องทางการติดตามและแจ้งเบาะแสข่าวปลอม"
FOOTER_START = "LINE : @antifakenewscenter"  # the site's contact block closes every post from here on

# The scammer phones the victim -> "call", anything else -> "text".
# Only phrases about being called count; a bare "โทรศัพท์" or a report hotline does not.
CALL = (r"คอลเซ็นเตอร์|ทางโทรศัพท์|ผ่านโทรศัพท์|(?i:\bcall\s*center\b)|รับสาย|ได้รับสาย"
        r"|โทรศัพท์(?:มา|หา|ติดต่อ|สอบ|แจ้ง|ถาม|ติดตาม)"
        r"|โทร(?:มา|หา|ไป|แจ้ง|ติดต่อ|สอบ|ถาม|ตรวจสอบ|หลอก|ขู่|เข้า)")
HOTLINE = r"(?i:call\s*center)\s*\d{3,4}|สายด่วน\s*\d+"  # e.g. "Call Center 1284", not a scam call


def clean_html(fragment):
    soup = BeautifulSoup(fragment, "html.parser")
    for tag in soup(["script", "style", "iframe", "figure", "img"]):
        tag.decompose()
    lines = (re.sub(r"[ \t]+", " ", line).strip() for line in soup.get_text("\n").replace("\xa0", " ").splitlines())
    lines = [line for line in lines if line]
    # drop the contact footer (LINE, website, Facebook, X, TikTok, IG) and its heading
    for i, line in enumerate(lines):
        if line.startswith(FOOTER_START):
            lines = lines[:i]
            if lines and lines[-1] == FOOTER_HEADING:
                lines.pop()
            break
    return "\n".join(lines)


def label_channel(text):
    return "call" if re.search(CALL, re.sub(HOTLINE, "", text)) else "text"


rows = []
for p in raw_posts:  # all fetched under #มิจฉาชีพ; keep those that also carry a target hashtag
    matched = [name for name in TARGET_TAGS if tag_ids[name] in p["tags"]]
    if matched:
        rows.append({
            "post_id": p["id"],
            "date": p["date"],
            "hashtag": ", ".join(matched),  # both tags -> one row, e.g. "อาชญากรรมออนไลน์, ข่าวปลอม"
            "title": html.unescape(BeautifulSoup(p["title"]["rendered"], "html.parser").get_text()).strip(),
            "content": clean_html(p["content"]["rendered"]),
        })

df = pd.DataFrame(rows)
# real datetime64 column, so it sorts chronologically (a DD/MM/YYYY string would sort by day first)
df["date"] = pd.to_datetime(df["date"]).dt.normalize()
df["Channel"] = (df["title"] + "\n" + df["content"]).map(label_channel)
df = df.sort_values("date", ascending=False, ignore_index=True)
df

In [ ]:
# drop posts with no title or no content (e.g. image-only posts): there is nothing to label in them
empty = (df["title"].str.strip() == "") | (df["content"].str.strip() == "")
if empty.any():
    print(f"dropping {empty.sum()} posts with no title or content: {df.loc[empty, 'post_id'].tolist()}")
df = df[~empty].reset_index(drop=True)

# test run: keep only the newest MAX_POSTS posts (df is sorted newest first)
if MAX_POSTS is not None:
    df = df.head(MAX_POSTS)
    print(f"test run: kept the {len(df)} newest posts")

# check the scrape now, before ORG and Qwen labelling, so a bad scrape stops here and costs no API calls
assert df["post_id"].is_unique
assert df["date"].notna().all()
assert df["hashtag"].str.contains("|".join(TARGET_TAGS)).all()
assert not df["content"].str.contains(FOOTER_START, regex=False).any()
print(f"{len(df)} posts passed the checks")

## Organization column

The `ORG` column names the Thai organization each post says the scammer pretended to be. It is
matched against `data/thai_organizations.csv`, a list of Thai government agencies, ministries and
banks with their **full names and abbreviations**. That list is built by `build_organizations.ipynb`
(from DGA open data, Thai Wikipedia and Wikidata); run that notebook first, and again only when you
want a fresh list.

A post is matched on any alias of an
organization (full name, abbreviation, `ธ.`/`แบงก์` + bank name, English name) after removing spaces,
because Thai spacing is irregular. The title is searched first. If it names no organization, the most
mentioned one in the content is used, ignoring the police, since almost every post tells the reader to
report to them. Limits: private companies (HomePro, Lotus's, Singha ...) and local bodies are not in
the list, so those posts get an empty `ORG`, and a match means the post *mentions* the
organization, not strictly that the scammer impersonated it.

In [ ]:
# the organization list is built by build_organizations.ipynb; here it is only read
# (dtype=str and keep_default_na=False so an empty abbreviation stays "" rather than NaN)
ORG_PATH = Path("data/thai_organizations.csv")
orgs = pd.read_csv(ORG_PATH, dtype=str, encoding="utf-8-sig", keep_default_na=False)

has_thai = lambda s: bool(re.search(r"[฀-๿]", s))  # a Thai alias needs more characters than an English one

print(f"{len(orgs)} organizations from {ORG_PATH}; {(orgs['abbr_en'] != '').sum()} with an English abbreviation")

In [ ]:
def squash(text):
    """Drop Thai spacing (it is irregular) but keep the gap between two Latin words."""
    text = text.replace("​", "")

    def gap(m):
        before = text[m.start() - 1] if m.start() else ""
        after = text[m.end()] if m.end() < len(text) else ""
        return " " if re.match(r"[A-Za-z0-9]", before) and re.match(r"[A-Za-z0-9]", after) else ""

    return re.sub(r"\s+", gap, text)


# colloquial names the lists do not carry: canonical Thai name -> extra aliases
MANUAL_ALIASES = {
    "สำนักงานตำรวจแห่งชาติ": ["ตำรวจ", "ตร."],
    "กรมสอบสวนคดีพิเศษ": ["ดีเอสไอ"],
    "ธนาคารแห่งประเทศไทย": ["แบงก์ชาติ"],
    "ธนาคารทหารไทยธนชาต": ["ทีทีบี"],
}
# ordinary words that must not stand in for an organization on their own
AMBIGUOUS_BANK_BARE = {"กรุงเทพ", "ไทย", "เครดิต"}  # keep "ธนาคาร"/"ธ." in front
GENERIC_COMPANY_BASE = {"ขนส่ง", "การบิน"}          # "บริษัท ขนส่ง จำกัด" minus บริษัท/จำกัด
BLOCKED_ALIASES = {"SME"}                           # loan product word, not the bank
IGNORE_IN_CONTENT = {"สำนักงานตำรวจแห่งชาติ"}        # posts end with "report to the police"


def org_aliases(row):
    name = row.name_th
    base = re.sub(r"\s*\((?:มหาชน|องค์การมหาชน)\)|\s*จำกัด|^บริษัท\s*", "", name).strip()
    out = {name, *MANUAL_ALIASES.get(name, []), *filter(None, row.abbr_en.split("|"))}
    if len(squash(base)) >= 6 and base not in GENERIC_COMPANY_BASE:
        out.add(base)
    if base.startswith("ธนาคาร"):
        rest = base[len("ธนาคาร"):].strip()
        out |= {"ธ." + rest, "แบงก์" + rest}
        if len(rest) >= 4 and rest not in AMBIGUOUS_BANK_BARE:
            out.add(rest)
    for a in filter(None, row.abbr_th.split("|")):
        out |= {a, re.sub(r"^สำนักงาน\s*", "", a)}  # "สำนักงาน ก.ล.ต." is also written "ก.ล.ต."
    # Thai abbreviations need >= 4 characters incl. dots, English ones >= 3, or they hit ordinary words
    return {a for a in out if a not in BLOCKED_ALIASES and len(squash(a)) >= (4 if has_thai(a) else 3)}


ALIAS_TO_ORG = {}
for row in orgs.itertuples():
    for alias in org_aliases(row):
        ALIAS_TO_ORG.setdefault(squash(alias), row.name_th)  # on a clash the first listed organization wins

# one pattern, longest alias first so the longest name wins; English aliases must be whole words
ALIAS_RE = re.compile("|".join(
    re.escape(a) if has_thai(a) else rf"(?<![A-Za-z]){re.escape(a)}(?![A-Za-z])"
    for a in sorted(ALIAS_TO_ORG, key=len, reverse=True)
))


def org_mentions(text):
    """Organizations named in the text, in order of appearance."""
    return [ALIAS_TO_ORG[m.group()] for m in ALIAS_RE.finditer(squash(text))]


def label_organization(title, content):
    in_title = dict.fromkeys(org_mentions(title))  # unique, in order
    if in_title:
        return ", ".join(in_title)
    counts = Counter(org for org in org_mentions(content) if org not in IGNORE_IN_CONTENT)
    return counts.most_common(1)[0][0] if counts else ""


df["ORG"] = [label_organization(t, c) for t, c in zip(df["title"], df["content"])]
labelled = df.loc[df["ORG"] != "", "ORG"]
print(f"{len(labelled)}/{len(df)} posts have an organization")
labelled.str.split(", ").explode().value_counts().head(20)

## Scam type and channel columns (labelled by Qwen)

Two columns are filled by Qwen, called through the university's OpenAI-compatible API
(`llm_labeler.py`):

- `scam_type`: one of the five Callas scam types (`fake_authority`, `romance`, `investment`,
  `phishing_otp`, `lottery`), or empty when the post describes no scam (an overview of several types,
  a fact-check of a rumour).
- `method_channel`: where the scam happens (see "Channel column" below).

The model gets the whole of `scam_type_definitions.md` as its instructions.

In [ ]:
from llm_labeler import FIELDS, label_all

# 50 posts per API call, one call after another; returns scam_type and method_channel + llm_error
# (a run that stops early resumes from data/llm_labels_checkpoint.jsonl)
labels = label_all(df)
df[FIELDS] = labels[FIELDS]

print(f"{labels['llm_error'].sum()} posts failed to label")
df["scam_type"].replace("", "(none)").value_counts()

## Channel column

`method_channel` says where each scam happens: เพจเฟซบุ๊ก, กลุ่มเฟซบุ๊ก, TikTok, LINE, SMS, โทรศัพท์,
เว็บไซต์, อีเมล, เอกสาร ... (comma separated). It comes from the same Qwen call as `scam_type` (cell
above); channels outside the fixed list are dropped, and posts with no `scam_type` have it empty.
A post whose `method_channel` lists โทรศัพท์ is a phone scam (`call_suitable` = `call`, see "Scenario columns").

In [ ]:
df[["title", "method_channel"]].head()

## Scenario columns (labelled by Qwen)

A second Qwen pass, only on posts with a `scam_type`, describes each scam the way the call simulator
(`modules/data_pipeline`) needs it. Field definitions are in `prompts/scenario_system.txt`.

| Column | What it holds |
|---|---|
| `call_suitable` | `call` (the scammer phones; always `call` when `method_channel` lists โทรศัพท์), `convertible` (could be told in a call) or `no` |
| `summary` | one English line naming the pattern |
| `caller_role` | who the scammer claims to be, in English |
| `goal` | what the scammer wants by the end, in English, with slots such as `{MONEY}` |
| `tactics` | the scam's steps in order, in English, joined with ` \| ` |
| `phrases` | Thai lines the post quotes from the scammer, with slots, joined with ` \| ` |
| `scenario_key` | the matching scenario in the simulator's `scenarios.yaml`, or `new` |
| `scenario_group` | a short name shared by posts with the same new situation |
| `source_url` | the post's link, built from `post_id` (no Qwen) |

Real numbers, links and LINE IDs are written as the simulator's slots. Posts with no `scam_type` have
these columns empty (except `source_url`).

In [ ]:
from llm_labeler import add_scenario_columns

# 20 posts per API call, only posts with a scam_type; adds the scenario columns and source_url
# (a run that stops early resumes from data/llm_scenarios_checkpoint.jsonl)
df = add_scenario_columns(df)
df["call_suitable"].replace("", "(no scam)").value_counts()

In [ ]:
print(f"{len(raw_posts)} posts under #{BASE_TAG} -> {len(df)} kept")
display(df["hashtag"].value_counts())
df["Channel"].value_counts()

In [ ]:
from llm_labeler import save_dataset

# the data was checked right after scraping (see the cell after the scrape)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
# writes OUT_PATH (data/antifakenews_scam.csv, or ..._test.csv on a test run):
# UTF-8 with BOM (Thai shows in Excel), the date as DD/MM/YYYY
save_dataset(df, OUT_PATH)
print(f"saved {len(df)} posts to {OUT_PATH}")

# read it back with the same format so the column is datetime again and sortable
# keep_default_na=False so empty labels show as "" instead of NaN
pd.read_csv(OUT_PATH, parse_dates=["date"], date_format="%d/%m/%Y", keep_default_na=False).head()

In [22]:
# only the lottery and romance posts, with the columns the call simulator uses
SIMULATOR_COLUMNS = [
    "scam_type", "call_suitable",                              # choosing which posts to use
    "date", "source_url", "summary", "tactics", "phrases",     # the wiki note
    "caller_role", "ORG", "goal", "scenario_group",            # the role card
]
lottery_romance = df[df["scam_type"].isin(["lottery", "romance"])][SIMULATOR_COLUMNS]
print(f"{len(lottery_romance)} posts: {lottery_romance['scam_type'].value_counts().to_dict()}")
lottery_romance

1 posts: {'lottery': 1}


,scam_type,call_suitable,date,source_url,summary,tactics,phrases,caller_role,ORG,goal,scenario_group
8,lottery,no,2026-10-20,https://www.antifakenewscenter.com/?p=93751,Scammer sends a fake 7-Eleven survey link prom...,Send a LINE message with a link to a fake surv...,กจิกรรมว่ันออกพรรษา แลกรัับเงินรางวัล {MONEY},7-Eleven marketing staff,,get the receiver to click a malicious link and...,fake_survey_prize
